In [1]:
!pip install transformers datasets
!pip install torch

In [2]:
import pandas as pd
import random
from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter

In [3]:
dataset = load_dataset("argilla/reward-model-data-falcon")


def create_dataset(row):
  if row['choose-best']['value'][0]==2:
    row['response-1'],row['response-2']=row['response-2'],row['response-1']
  return row

dataset=dataset.map(lambda x: create_dataset(x))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-862148f61ac741(…):   0%|          | 0.00/5.21M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7401 [00:00<?, ? examples/s]

Map:   0%|          | 0/7401 [00:00<?, ? examples/s]

In [4]:
def loss_function(preferred_response_reward, alternate_response_reward):
    return -torch.mean(torch.log(torch.sigmoid(alternate_response_reward - preferred_response_reward)))

In [5]:
from transformers import GPT2Model, GPT2Tokenizer

class GPT2RewardModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = GPT2Model.from_pretrained('gpt2')
        self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.regression_head = torch.nn.Linear(768, 1)

    def forward(self, context, response):

        entire_text = context + response
        context_dict = self.tokenizer(
            '<|startoftext|>' + entire_text + '<|endoftext|>'
        )

        input_ids = torch.tensor(context_dict.input_ids)
        attention_mask = torch.tensor(context_dict.attention_mask)


        gpt2_outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        all_output_vectors = gpt2_outputs.last_hidden_state
        last_output_vector = all_output_vectors[-1]


        last_output_vector = last_output_vector.unsqueeze(0)
        reward = self.regression_head(last_output_vector)

        return reward
model = GPT2RewardModel()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
def train(epochs=10):

    # Create the optimizer
    optimizer = torch.optim.Adam(
        model.parameters(), lr=1e-5)


    batch_idx = 0
    # Train the model
    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}")
        for batch in tqdm(dataset):

            # Get the data
            prompt, preferred_reponse, alternate_response,choose,external_id = batch
            preferred_response_reward = model(prompt, preferred_reponse)
            alternate_response_reward = model(prompt, alternate_response)

            loss = loss_function(preferred_response_reward, alternate_response_reward)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Zero the gradients
            optimizer.zero_grad()

            # Log the loss
            print(f"Loss: {loss.item()}", batch_idx)
            batch_idx += 1


train()

Epoch 1


100%|██████████| 1/1 [00:03<00:00,  3.16s/it]


Loss: 0.6836042404174805 0
Epoch 2


100%|██████████| 1/1 [00:01<00:00,  1.82s/it]


Loss: 0.6693341135978699 1
Epoch 3


...

Loss: 0.5224211812019348 7
Epoch 9


100%|██████████| 1/1 [00:02<00:00,  2.67s/it]


Loss: 0.4776158630847931 8
Epoch 10


100%|██████████| 1/1 [00:01<00:00,  1.86s/it]

Loss: 0.42644956707954407 9
